# Reschedule French Deck By Frequency

This notebook connects to Anki via AnkiConnect, ranks cards in the `French` deck by local frequency lists, and reorders the deck so more frequent items come first.

Safe default behavior:
- preview everything first
- reorder `new` cards by frequency
- leave non-new cards alone unless you explicitly enable review rescheduling

In [ ]:
import csv
import html
import json
import re
import unicodedata
import urllib.request
from pathlib import Path

import pandas as pd
from bs4 import BeautifulSoup

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

ANKI_CONNECT_URL = "http://127.0.0.1:8765"
ANKI_CONNECT_VERSION = 6
WRITE_BATCH_SIZE = 500


def invoke(action, **params):
    payload = json.dumps({"action": action, "version": ANKI_CONNECT_VERSION, "params": params}).encode("utf-8")
    req = urllib.request.Request(ANKI_CONNECT_URL, data=payload, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    if data.get("error"):
        raise RuntimeError(f"AnkiConnect error in {action}: {data['error']}")
    return data.get("result")


def batched(items, size):
    for i in range(0, len(items), size):
        yield items[i:i + size]


def action_supported(action, **params):
    payload = json.dumps({"action": action, "version": ANKI_CONNECT_VERSION, "params": params}).encode("utf-8")
    req = urllib.request.Request(ANKI_CONNECT_URL, data=payload, headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode("utf-8"))
    return data.get("error") != "unsupported action"


print("Connected AnkiConnect version:", invoke("version"))

In [ ]:
DECK_NAME = "French"
DRY_RUN = True

# Safe default: only reorder new cards.
# If your AnkiConnect build does not support direct new-card repositioning,
# the notebook falls back to setDueDate(), which converts new cards into
# scheduled review cards in frequency order.
# If you set this to True, existing non-new cards will also be assigned
# new due dates one by one in frequency order, which is much more invasive.
RESCHEDULE_EXISTING_CARDS = False
NEW_CARD_START_DAY = 0
EXISTING_CARD_START_DAY = 0

FREQUENCY_SOURCES = [
    {
        "path": Path("../data/french-frequency/fr_full.txt"),
        "format": "word_count",
    },
]

FIELD_PRIORITY = [
    "French",
    "Word",
    "Expression",
    "Lemma",
    "Term",
    "Front",
    "Vocabulary",
    "Vocab",
]

print(f"Deck: {DECK_NAME}")
print(f"DRY_RUN: {DRY_RUN}")
print(f"RESCHEDULE_EXISTING_CARDS: {RESCHEDULE_EXISTING_CARDS}")

In [ ]:
TAG_RE = re.compile(r"<[^>]+>")
BRACKET_RE = re.compile(r"\[[^\]]*\]|\([^\)]*\)")


def normalize_text(text):
    text = html.unescape(text or "")
    text = BeautifulSoup(text, "html.parser").get_text(" ", strip=True)
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("’", "'")
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


def candidate_forms(text):
    text = normalize_text(text)
    if not text:
        return []

    forms = []

    def add(value):
        value = normalize_text(value)
        if value and value not in forms:
            forms.append(value)

    add(text)
    add(text.split("\n", 1)[0])
    add(BRACKET_RE.sub("", text))

    first_chunk = re.split(r"[;,/|]", text, maxsplit=1)[0]
    add(first_chunk)

    tokens = re.findall(r"[\w'’À-ÿ-]+", text)
    if tokens:
        add(tokens[0])
        if len(tokens) >= 2:
            add(" ".join(tokens[:2]))

    return forms


def load_frequency_words(source):
    path = source["path"]
    if not path.exists():
        raise FileNotFoundError(path)

    if source["format"] == "word_count":
        words = []
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.strip()
            if not line:
                continue
            word = normalize_text(line.rsplit(" ", 1)[0])
            if word:
                words.append(word)
        return words

    if source["format"] == "word_list":
        words = []
        for line in path.read_text(encoding="utf-8").splitlines():
            word = normalize_text(line)
            if word:
                words.append(word)
        return words

    if source["format"] == "ranked_csv":
        df = pd.read_csv(path)
        words = [normalize_text(value) for value in df[source["word_column"]].fillna("")]
        return [word for word in words if word]

    raise ValueError(f"Unsupported source format: {source['format']}")


def build_frequency_rank_map(sources):
    rank_map = {}
    next_rank = 1
    for source in sources:
        words = load_frequency_words(source)
        print(f"Loaded {len(words)} words from {source['path']}")
        for word in words:
            if word not in rank_map:
                rank_map[word] = next_rank
                next_rank += 1
    return rank_map


frequency_rank = build_frequency_rank_map(FREQUENCY_SOURCES)
UNKNOWN_RANK = len(frequency_rank) + 1_000_000
print(f"Combined unique frequency words: {len(frequency_rank)}")

In [ ]:
all_card_ids = invoke("findCards", query=f'deck:"{DECK_NAME}"')
new_card_id_set = set(invoke("findCards", query=f'deck:"{DECK_NAME}" is:new'))
suspended_card_id_set = set(invoke("findCards", query=f'deck:"{DECK_NAME}" is:suspended'))

print(f"All cards: {len(all_card_ids)}")
print(f"New cards: {len(new_card_id_set)}")
print(f"Suspended cards: {len(suspended_card_id_set)}")

In [ ]:
cards = []
for batch in batched(all_card_ids, WRITE_BATCH_SIZE):
    cards.extend(invoke("cardsInfo", cards=batch))

print(f"Fetched card info for {len(cards)} cards")

if cards:
    sample_fields = sorted(cards[0]["fields"].keys())
    print("Sample field names:", sample_fields)

In [ ]:
def choose_primary_field(fields):
    field_names = list(fields.keys())
    for candidate in FIELD_PRIORITY:
        if candidate in fields:
            return candidate
    return field_names[0] if field_names else None


ranked_cards = []
for card in cards:
    fields = card.get("fields", {})
    field_name = choose_primary_field(fields)
    field_value = fields[field_name]["value"] if field_name else ""
    candidates = candidate_forms(field_value)

    matched_word = None
    matched_rank = UNKNOWN_RANK
    for candidate in candidates:
        rank = frequency_rank.get(candidate)
        if rank is not None and rank < matched_rank:
            matched_rank = rank
            matched_word = candidate

    ranked_cards.append(
        {
            "card_id": card["cardId"],
            "note_id": card["note"] if "note" in card else card.get("noteId"),
            "deck_name": card.get("deckName", ""),
            "field_name": field_name,
            "field_value": normalize_text(field_value),
            "matched_word": matched_word,
            "rank": matched_rank,
            "is_new": card["cardId"] in new_card_id_set,
            "is_suspended": card["cardId"] in suspended_card_id_set,
        }
    )

ranked_df = pd.DataFrame(ranked_cards).sort_values(
    by=["rank", "matched_word", "field_value", "card_id"],
    ascending=[True, True, True, True],
).reset_index(drop=True)

ranked_df.head(20)

In [ ]:
matched_df = ranked_df[ranked_df["rank"] < UNKNOWN_RANK]
unmatched_df = ranked_df[ranked_df["rank"] >= UNKNOWN_RANK]
new_sorted_ids = ranked_df.loc[
    ranked_df["is_new"] & ~ranked_df["is_suspended"],
    "card_id",
].tolist()
existing_sorted_ids = ranked_df.loc[
    ~ranked_df["is_new"] & ~ranked_df["is_suspended"],
    "card_id",
].tolist()

print(f"Matched cards: {len(matched_df)} / {len(ranked_df)}")
print(f"Unmatched cards: {len(unmatched_df)}")
print(f"New cards to reorder: {len(new_sorted_ids)}")
print(f"Existing non-suspended cards: {len(existing_sorted_ids)}")

print("\nTop 20 ranked cards:")
display(ranked_df[["card_id", "field_name", "field_value", "matched_word", "rank", "is_new"]].head(20))

print("\nFirst 20 unmatched cards:")
display(unmatched_df[["card_id", "field_name", "field_value", "is_new"]].head(20))

In [ ]:
if DRY_RUN:
    print("DRY_RUN is True. No changes were written.")
else:
    can_reposition_new = action_supported(
        "repositionNewCards",
        cards=[1],
        startingPosition=1,
        step=1,
        randomize=False,
        shift=True,
    )

    if new_sorted_ids:
        if can_reposition_new:
            invoke(
                "repositionNewCards",
                cards=new_sorted_ids,
                startingPosition=1,
                step=1,
                randomize=False,
                shift=True,
            )
            print(f"Repositioned {len(new_sorted_ids)} new cards by frequency.")
        else:
            print("repositionNewCards is not supported by this AnkiConnect build.")
            print("Falling back to setDueDate(), which converts new cards into scheduled review cards.")
            for offset, card_id in enumerate(new_sorted_ids, start=NEW_CARD_START_DAY):
                invoke("setDueDate", cards=[card_id], days=str(offset))
            print(f"Assigned due dates to {len(new_sorted_ids)} formerly-new cards by frequency.")
    else:
        print("No new cards to reorder.")

    if RESCHEDULE_EXISTING_CARDS:
        print(f"Rescheduling {len(existing_sorted_ids)} existing cards one by one...")
        for offset, card_id in enumerate(existing_sorted_ids, start=EXISTING_CARD_START_DAY):
            invoke("setDueDate", cards=[card_id], days=str(offset))
        print("Finished rescheduling existing cards.")
    else:
        print("Existing cards were left untouched. Set RESCHEDULE_EXISTING_CARDS = True to change them too.")